In [0]:
-- TASK 1A
USE CATALOG data;
USE SCHEMA default;

/* THE QUERY BELOW IS A MERGE STATEMENT THAT MERGES THE ACCOUNTS AND TRANSACTIONS TABLES INTO A SINGLE TABLE CALLED MASTER_FILE.*/ 
WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM transactions t
        INNER JOIN accounts a  ON t.account_id = a.account_id
        INNER JOIN customers c ON a.customer_id = c.customer_id
    )

SELECT 
    customer_id, 
    account_type, 
    SUM(balance) AS total_balance -- CALCULATES THE SUM OF BALANCE AND COLUMN NAME SET TO TOTAL BALANCE
FROM 
    Master_File -- CTE TEMPORARY TABLE 
GROUP BY 
    customer_id,  
    account_type; 
    /*
    -GROUPBY- GROUPS BY ACCOUNT TYPE - THIS ENSURES THAT THE SUM OF BALANCE IS CALCULATED FOR EACH ACCOUNT TYPE
    -GROUPBY- GROUPS BY ACCOUNT TYPE - THIS ENSURES THAT THE SUM OF BALANCE IS CALCULATED FOR EACH ACCOUNT TYPE
    */


In [0]:
-- TASK 1B
/*New accounts created in the last year. Max_Date is being used as Current_Date*/
WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM data.default.transactions t
        INNER JOIN data.default.accounts a  ON t.account_id = a.account_id
        INNER JOIN data.default.customers c ON a.customer_id = c.customer_id
    )
SELECT 
    customer_id, 
    first_name, 
    last_name, 
    account_id, 
    opening_date 
FROM 
    Master_File
WHERE 
    opening_date >= (SELECT MAX(opening_date) FROM Master_File) - INTERVAL 1 YEAR
GROUP BY 
    customer_id, 
    first_name, 
    last_name, 
    account_id, 
    opening_date;


In [0]:
-- TASK 1C
/*Top 5 customer based on Total balance*/
WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM data.default.transactions t
        INNER JOIN data.default.accounts a  ON t.account_id = a.account_id
        INNER JOIN data.default.customers c ON a.customer_id = c.customer_id
    )
/*Top 5 Customers based on balance*/
SELECT 
    first_name, 
    last_name, 
    SUM(balance) AS total_balance -- Alias renamed to total_balance
FROM 
    master_file -- CTE TEMPORARY TABLE
GROUP BY 
    customer_id, 
    first_name, 
    last_name
ORDER BY 
    total_balance DESC -- High to Low Order
LIMIT 5; -- Contrain to 5 rows/instances

In [0]:
-- TASK 2A 
/*The aim of this query is to capture all transactions that are withdrawals and have an amount greater than $500 in the last 30 days */
WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM data.default.transactions t
        INNER JOIN data.default.accounts a  ON t.account_id = a.account_id
        INNER JOIN data.default.customers c ON a.customer_id = c.customer_id
    )
SELECT 
    account_id, 
    customer_id, 
    first_name, 
    last_name, 
    transaction_date, 
    amount
FROM 
    Master_File -- CTE TEMPORARY TABLE
WHERE 
    transaction_type = 'Withdrawal' -- Filter for withdrawals 
    AND amount > 500 -- Filter for amounts greater than $500
    AND transaction_date >= (SELECT MAX(transaction_date) FROM Master_File) - INTERVAL 30 DAYS; 
    -- Filter for transactions in the last 30 days
   

In [0]:
-- TASK 2B
/*
The purpose of this function is to calculate the number of deposits produced in the last 6 months 
Max_Transaction_Date is being used as Current_Date
*/
WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM data.default.transactions t
        INNER JOIN data.default.accounts a  ON t.account_id = a.account_id
        INNER JOIN data.default.customers c ON a.customer_id = c.customer_id
    )
SELECT 
    customer_id, 
    COUNT(transaction_type) AS total_deposits -- The number of transactions that are deposits
FROM 
    Master_File -- CTE TABLE
WHERE 
    transaction_type = 'Deposit' -- Filter for deposits
    AND transaction_date >= (SELECT MAX(transaction_date) FROM Master_File) - INTERVAL 6 MONTHS 
    -- Filter for transactions in the last 6 months
GROUP BY 
   customer_id; -- Group by customer_id

In [0]:
--Task 2C
/*
Capture Running Balance - Simulation - 
The aim of this is to get a running balance for each account -  Using deposits and withdrawals tags - to simulate the balance of each account
*/
WITH Raw_Data AS (
    SELECT 
        t.*, 
        c.customer_id,
        c.first_name, 
        c.last_name,  
        c.date_of_birth, 
        c.address, 
        c.city, 
        c.state, 
        c.zip,  
        a.balance, 
        a.account_type, 
        a.opening_date,
        -- Look ahead at the history of the account to find the very first transaction amount
        FIRST_VALUE(t.amount) OVER (
            PARTITION BY t.account_id 
            ORDER BY t.transaction_date ASC, t.transaction_id ASC
        ) AS first_transaction_amount
    FROM data.default.transactions t
    INNER JOIN data.default.accounts a  ON t.account_id = a.account_id
    INNER JOIN data.default.customers c ON a.customer_id = c.customer_id
),

Master_File AS (
    -- Step 2: Calculate historical anchors per account partition
    SELECT 
        rd.*,
        -- Generate an index (1, 2, 3...) sorted oldest to newest per account
        ROW_NUMBER() OVER (
            PARTITION BY rd.account_id 
            ORDER BY rd.transaction_date ASC, rd.transaction_id ASC
        ) AS tx_sequence,
        -- Grab the absolute first transaction amount for this account
        FIRST_VALUE(rd.amount) OVER (
            PARTITION BY rd.account_id 
            ORDER BY rd.transaction_date ASC, rd.transaction_id ASC
        ) AS first_amount
    FROM 
        Raw_Data rd
)

-- Step 3: QUERY CTE TABLE
SELECT 
    account_id, 
    transaction_id, 
    transaction_date,
    amount,
    transaction_type,
    SUM(CASE 
        -- 1. DEPOSITS: Always add to the running balance
        WHEN transaction_type = 'Deposit' THEN ABS(amount)
        
        -- 2. WITHDRAWALS: Checked against the sequence and first transaction rules
        WHEN transaction_type = 'Withdrawal' THEN 
            CASE 
                -- If it IS NOT the first transaction (sequence > 1), strictly force a deduction
                WHEN tx_sequence > 1 THEN -ABS(amount)
                -- If it IS the first transaction, THEN OVERDRAFT
                WHEN first_amount < 0 THEN amount 
                ELSE -ABS(amount) 
            END
            
        -- 3. PAYMENTS: Negative values add (Refunds), positive values subtract (Debits)
        WHEN transaction_type = 'Payment' THEN 
            CASE 
                WHEN amount < 0 THEN ABS(amount)  -- REFUNDS ARE SEEN HAS NEGATIVE PAYMENTS
                ELSE -ABS(amount)                 -- TURNED NEGATIVE TO ENFORCE PAYMENT TRANSACTIONS
            END
            
        -- 4. TRANSFERS: Pass value directly based on its ledger sign (+/-)
        WHEN transaction_type = 'Transfer' THEN amount 
        
        ELSE 0 
    END) OVER (
        PARTITION BY account_id 
        ORDER BY transaction_date ASC, transaction_id ASC  
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_balance
FROM 
    Master_File
ORDER BY 
    account_id, 
    transaction_date ASC, 
    transaction_id ASC;

In [0]:
--  TASK 3A
-- 1. Get the reference date (max date in dataset)
WITH DateAnchor AS (
    SELECT MAX(transaction_date) AS max_date FROM transactions
),
/* 2. GET THE AVERAGE AMOUNT BY CUSTOMER_ID(using the accounts table to link them)
    A) -- JOIN ACCOUNTS TABLE WITH TRANSACTIONS TABLE ON ACCOUNT_ID
    B) -- CROSS JOIN WITH DATEANCHOR TABLE TO GET MAX DATE
    C) -- FILTER FOR TRANSACTIONS WITHIN LAST YEAR(USE MAX DATE HAS A CURRENT DATE AND SUBTRACT 1 YEAR )
    D) -- GROUPBY ACCOUNT CUSTOMER ID

    */
Customer_AVG AS (
    SELECT 
        a.customer_id,
        AVG(t.amount) AS avg_transaction_amount,
        AVG(a.balance) AS avg_balance 
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id 
    CROSS JOIN DateAnchor d 
    WHERE t.transaction_date >= d.max_date - INTERVAL 1 YEAR 
    GROUP BY a.customer_id
),

/* 3. GET THE AVERAGE BALANCE BY CUSTOMER_ID(using the accounts table to link them)
    A) -- GROUPBY ACCOUNT CUSTOMER ID
    B) -- GET AVERAGE OF BALANCE
    */

Customer_Balance_Stats AS (
    SELECT 
        customer_id,
        AVG(balance) AS avg_balance 
    FROM accounts
    GROUP BY customer_id 
)
/*
4. QUERY ALL CTE TABLES ABOVE
 A) REPLACE NULLS WITHIN AVERAGE_TRANSACTION_AMOUNT TO ($0)
 B) JOIN BOTH CTE TABLES TOGETHER
 C) ORDER BY AVERAGE BALANCE DESCENDING 
*/

SELECT 
    b.customer_id,
    COALESCE(t.avg_transaction_amount, 0) AS avg_transaction_amount,
    COALESCE(t.avg_balance, 0) AS avg_balance
    
FROM Customer_Balance_Stats b
LEFT JOIN Customer_AVG t ON b.customer_id = t.customer_id -- JOIN BOTH CTE TABLE TOGETHER
ORDER BY b.customer_id ASC;

In [0]:
--TASK 3B
/*HIGHLIGHT */

WITH Master_File AS (
    SELECT 
            t.*, 
            c.customer_id,
            c.first_name, 
            c.last_name,  
            c.date_of_birth, 
            c.address, 
            c.city, 
            c.state, 
            c.zip,  
            a.balance, 
            a.account_type, 
            a.opening_date 
        FROM transactions t
        INNER JOIN accounts a  ON t.account_id = a.account_id
        INNER JOIN customers c ON a.customer_id = c.customer_id
    )
/* THE PURPOSE OF THIS QUERY IS TO DETERMINE THE MOST AMOUNT TRANSACTION BY CUSTOMER IN THE LAST 3 MONTHS
A) FILTER FOR TRANSACTIONS WITHIN LAST 3 MONTHS (MAX TRANSACTION_DATE IS BEING USED AS CURRENT DATE)
B) GROUPBY first_name, last_name and customer_id
C) ORDER BY COUNT(*) DESCENDING - HIGH TO LOW 
D) ADD A ROW CONSTRAING ON 1 - THIS WILL RETURN TOP 1*/ 

SELECT 
    customer_id,
    first_name,
    last_name, 
    COUNT(*) AS transaction_count 
FROM 
    Master_File
WHERE 
    transaction_date >= (SELECT MAX(transaction_date) FROM Master_File) - INTERVAL 3 MONTHS
GROUP BY 
    first_name,
    last_name,
    customer_id
ORDER BY 
    transaction_count DESC
LIMIT 1;


In [0]:
--TASK 4A
/*
A) CALCULATE THE SUM OF ALL TRANSACTIONS AMOUNT (DEPOSIT AND WITHDRAWALS), ANY OTHER TRANSACTIONS ($0)
B) JOIN BOTH THE ACCOUNT TABLE AND CTE (BASED ON TRANSACTION TABLE) ON account_id
C) FILTER FOR CASES WHERE ACCOUNT BALANCE -ROUNDED  2SF- IS NOT EQUAL TO ROUNDS TRANSACTION AMOUNTS

*/
WITH CalculatedBalances AS (
    -- Sum transactions per account, treating withdrawals as negative
    SELECT 
        account_id,
        SUM(CASE 
            WHEN transaction_type = 'Deposit' THEN ABS(amount) 
            WHEN transaction_type = 'Withdrawal' THEN -ABS(amount) 
            WHEN transaction_type = 'Transfer' THEN amount
            WHEN transaction_type = 'Payment' THEN 
            CASE 
                WHEN amount < 0 THEN ABS(amount)  -- Negative Payment = Refund (Adds to balance)
                ELSE -ABS(amount)                 -- Positive Payment = Standard Debit (Subtracts from balance)
            END
            ELSE 0 
        END) AS total_txn_sum
    FROM 
       data.default.transactions transactions
    GROUP BY 
        account_id
)
SELECT 
    a.account_id, 
    a.balance AS account_balance, 
    c.total_txn_sum AS calculated_balance
FROM 
    data.default.accounts a
JOIN 
    CalculatedBalances c ON a.account_id = c.account_id
WHERE 
    -- Filter for cases where the two values are not equal
    ROUND(a.balance, 2) != ROUND(c.total_txn_sum, 2);

In [0]:
--TASK 4B
/*
A) FILTER FOR NULL VALUES IN ADDRESS, DATE_OF_BIRTH AND ZIP
B) FILTER FOR ZIPCODES WITH INVALID ZIPCODES
C) THE CUSTOMER ADDESS, DATE_OF_BIRTH AND ZIPCODE IS SEARCHED FOR INVALID FIELDS
D) RETURN A STRING HIGHLIGHTING THE COLUMN WHERE THE VAULE IS INVALID


*/
SELECT 
    customer_id, 
    first_name, 
    last_name, 
    CONCAT_WS(', ',
        CASE WHEN address IS NULL OR address = '' THEN 'address' END,
        CASE WHEN date_of_birth IS NULL THEN 'date_of_birth' END,
        CASE 
            WHEN zip IS NULL 
                 OR LENGTH(zip) <= 4 
                 OR try_cast(zip AS BIGINT) IS NULL THEN 'invalid_zip' 
        END
    ) AS missing_or_invalid_fields
FROM 
    data.default.customers -- CUSTOMER TABLE
WHERE 
    address IS NULL OR address = '' OR
    date_of_birth IS NULL OR
    zip IS NULL OR 
    LENGTH(zip) <= 4;

In [0]:
--TASK 4C
SELECT 
    customer_id, 
    account_type, 
    COUNT(*) AS number_of_duplicates
FROM 
    accounts
GROUP BY 
    customer_id, 
    account_type
HAVING 
    COUNT(*) > 1;

In [0]:
-- TASK 4D

SELECT 
    transaction_id, 
    account_id, 
    transaction_type
FROM 
    transactions
WHERE 
    transaction_type NOT IN ('Deposit', 'Withdrawal', 'Payment', 'Transfer')
    OR transaction_type IS NULL;



In [0]:
-- TASK 4E
/* 
A) Highlight rows that are associated with Credit Account
B) Filter for negative balances (<0)

*/
SELECT 
    account_id, 
    customer_id, 
    account_type, 
    balance
FROM 
    accounts
WHERE 
    balance < 0 
    AND account_type != 'Credit';

